In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print('Length of dataset in characters: ', len(text))

Length of dataset in characters:  1115394


In [4]:
print(text[:300])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(len(chars))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for i,s in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l]

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
['h', 'i', 'i', ' ', 't', 'h', 'e', 'r', 'e']


In [7]:
data = torch.tensor(encode(text), dtype=torch.long)
data[:300]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57,  1, 47, 57,  1, 41, 

In [8]:
n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]
len(train_data)

1003854

In [9]:
block_size =8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
X = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = X[:t+1]
    target = y[t]
    print(f"When Input is: {context} the target: {target}")

When Input is: tensor([18]) the target: 47
When Input is: tensor([18, 47]) the target: 56
When Input is: tensor([18, 47, 56]) the target: 57
When Input is: tensor([18, 47, 56, 57]) the target: 58
When Input is: tensor([18, 47, 56, 57, 58]) the target: 1
When Input is: tensor([18, 47, 56, 57, 58,  1]) the target: 15
When Input is: tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
When Input is: tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [11]:
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split=='train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i: i+block_size] for i in ix])
    y = torch.stack([data[i+1: i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print(f"Inputs: {xb.shape}")
print(xb)
print(f"Outoutes: {yb.shape}")
print(yb)
print("-----------------------------")
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"When input is {context.tolist()} the target: {target}")

Inputs: torch.Size([4, 8])
tensor([[ 6,  0, 21,  1, 61, 53, 59, 50],
        [39, 50, 50, 43, 45, 47, 39, 52],
        [44,  1, 58, 46, 39, 58,  1, 21],
        [57,  8,  0,  0, 23, 21, 26, 19]])
Outoutes: torch.Size([4, 8])
tensor([[ 0, 21,  1, 61, 53, 59, 50, 42],
        [50, 50, 43, 45, 47, 39, 52, 41],
        [ 1, 58, 46, 39, 58,  1, 21,  1],
        [ 8,  0,  0, 23, 21, 26, 19,  1]])
-----------------------------
When input is [6] the target: 0
When input is [6, 0] the target: 21
When input is [6, 0, 21] the target: 1
When input is [6, 0, 21, 1] the target: 61
When input is [6, 0, 21, 1, 61] the target: 53
When input is [6, 0, 21, 1, 61, 53] the target: 59
When input is [6, 0, 21, 1, 61, 53, 59] the target: 50
When input is [6, 0, 21, 1, 61, 53, 59, 50] the target: 42
When input is [39] the target: 50
When input is [39, 50] the target: 50
When input is [39, 50, 50] the target: 43
When input is [39, 50, 50, 43] the target: 45
When input is [39, 50, 50, 43, 45] the target: 47
When

In [12]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embeding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embeding_table(idx) # (B,T,C) (4, 8, 65)
        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # It streches each row and becomes (32, 65)
            targets = targets.view(B*T) # previously it was (4, 8) and now it is also strected and became (32)
            loss = F.cross_entropy(logits, targets) # this expects something (B, C)
        return logits, loss
    
    def generate(self,idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # idx is originally (B, T) just like xb
            logits = logits[:, -1, :] # logits becomes (B, C) from (B,T,C)
            probs = F.softmax(logits, dim=1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape,'\n',loss)

print(decode(m.generate(torch.zeros((2,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65]) 
 tensor(4.7201, grad_fn=<NllLossBackward0>)
['\n', 'e', 'x', 'c', 'P', 'T', 'N', 'G', 'L', 'L', 'O', 'U', 'r', 'p', 'x', '&', 'V', 'P', '$', 'a', 'J', 'A', 'o', 'z', 'Y', 'H', 'G', 'T', 'N', 'G', 'R', 'e', 'l', '!', 'S', 't', 's', ';', 'F', 'Z', 'N', 'u', 'R', 'P', 'Q', 'i', ' ', 'Z', 'x', 'G', 'v', 'J', 'S', 'S', 'x', 's', 'A', 'H', 'w', 'L', 'A', 'T', 'w', 'e', 'l', 'q', 'G', 'E', 'c', 'J', 'Q', '?', 'h', ':', 'e', 'I', 'D', 'p', '&', 'r', '3', 'w', 'u', 'W', 'l', 'g', 'y', 'e', '!', 'Z', ',', 'P', '&', 'e', 'O', 'd', 'n', 'q', 'U', 'R', 'V']


In [13]:
# optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [14]:
batch_size=32
for steps in range(10000):

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluating loss

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.5998294353485107


In [15]:

print(''.join(decode(m.generate(torch.zeros((2,1), dtype=torch.long), max_new_tokens=1000)[0].tolist())))



OLLORIWhisextie hald w. grie tak'd the h ave.
I d thath
TRO:
Wirs f Vir

Hind.
O: mgom,
arde by wa athyodeit cea t'Thl herise ss ho 'sh tede, ld thive wn, yonan
She s teit me tcoroug.
LLO:
An,candivicowbs lid;
Fppallersel thth,
CESe imy GLOFiYOunespererighino tle a hopanl t t, herily!
D:
HAR: wothoden,
O;
SY NIS:
Wheno thichithis d urene s! ll qu le bupelkn dsdothanoto f tl
THADiey mHe matseIEa.
MEYo we wnovu winicond, thin he ght.
Tholr w tl pricobor ame,
Ad

Nay tone,
Dilanouino l tousxt c
Navon, sit are anor.



sl t ousired ndrainismpe toulor thout nit, SSowerFON inestweirinderehy foitilday CUCK:

T:
Gla byod wid thero thape grlascture.
Ge, woke pry erornouthagn rel itwnourouchedy asst ofar.
Cx;
Sacerorsin cous won er bisombeather ars towhe t, thysound!
n th hatuavyok:
BENut,
vartoknt? w$DUENI whar:
ORE shar; moure thlelod,
RI s h

BOr's mefeellershandiu,
Thyo, bre s?

Wine as:
Marouete, swideen

Warnk doungs PENUCalo thano g l I be. we hedo--vik?
ORCETh u,
Ty ano Goou bedoucooar

# Mathematic trick for Attention

In [16]:
# toy example
B, T, C = 4, 8, 2
x = torch.rand(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [17]:
# version 1: 
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = torch.mean(xprev, 0)

In [18]:
x[0]

tensor([[0.0284, 0.6631],
        [0.1715, 0.8923],
        [0.3227, 0.6377],
        [0.2784, 0.5361],
        [0.2363, 0.7109],
        [0.2346, 0.7749],
        [0.7789, 0.9318],
        [0.4421, 0.1672]])

In [19]:
xbow[0]

tensor([[0.0284, 0.6631],
        [0.0999, 0.7777],
        [0.1742, 0.7310],
        [0.2002, 0.6823],
        [0.2074, 0.6880],
        [0.2120, 0.7025],
        [0.2930, 0.7353],
        [0.3116, 0.6642]])

In [20]:
# using a nested for loop is too inefficient so rather we use a trick

In [21]:
# vesion 2: using matrix multiplication
wei = torch.tril(torch.ones(T, T))
wei = wei / torch.sum(wei, 1, keepdims=True)
xbow2 = wei @ x # (T, T) @ (B, T, C)  =>  (B, T, T) @ (B, T, C)  =>      (B, T, C)

In [22]:
xbow[0], xbow2[0]

(tensor([[0.0284, 0.6631],
         [0.0999, 0.7777],
         [0.1742, 0.7310],
         [0.2002, 0.6823],
         [0.2074, 0.6880],
         [0.2120, 0.7025],
         [0.2930, 0.7353],
         [0.3116, 0.6642]]),
 tensor([[0.0284, 0.6631],
         [0.0999, 0.7777],
         [0.1742, 0.7310],
         [0.2002, 0.6823],
         [0.2074, 0.6880],
         [0.2120, 0.7025],
         [0.2930, 0.7353],
         [0.3116, 0.6642]]))

In [23]:
torch.tril(torch.ones(3, 3))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [24]:
# version:3 there is another approach
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1) # first exponentiate and then and then take the sum according to the row
xbow3 = wei @ x
xbow[0], xbow3[0]

(tensor([[0.0284, 0.6631],
         [0.0999, 0.7777],
         [0.1742, 0.7310],
         [0.2002, 0.6823],
         [0.2074, 0.6880],
         [0.2120, 0.7025],
         [0.2930, 0.7353],
         [0.3116, 0.6642]]),
 tensor([[0.0284, 0.6631],
         [0.0999, 0.7777],
         [0.1742, 0.7310],
         [0.2002, 0.6823],
         [0.2074, 0.6880],
         [0.2120, 0.7025],
         [0.2930, 0.7353],
         [0.3116, 0.6642]]))

In [25]:
a = torch.tril(torch.ones(3, 3)) # this is a lower triangle matrix
a = a/torch.sum(a, 1, keepdim=True)
b = torch.rand(3, 2)
c = a@b

print("a = ", a)
print("--------")
print("b = ", b)
print("--------")
print("c = ",c)

a =  tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--------
b =  tensor([[0.4149, 0.1034],
        [0.7551, 0.7431],
        [0.1911, 0.8692]])
--------
c =  tensor([[0.4149, 0.1034],
        [0.5850, 0.4232],
        [0.4537, 0.5719]])


In [ ]:
# version 4: using Self Attention
B,T,C = 4,8,32
x=torch.rand(B, T, C)

# single attention head
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
wei = q @ k.transpose(-2, -1) * head_size**-0.5 # (B, T, head_size) @ (B, head_size, T) => (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros(T, T)
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1) # first exponentiate and then and then take the sum according to the row

v = value(x)
out = wei @ v
out.shape

torch.Size([4, 8, 32])

In [41]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5259, 0.4741, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3417, 0.3325, 0.3259, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2296, 0.1944, 0.2234, 0.3526, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1936, 0.1466, 0.1888, 0.2754, 0.1955, 0.0000, 0.0000, 0.0000],
        [0.1463, 0.1349, 0.1497, 0.2307, 0.1769, 0.1615, 0.0000, 0.0000],
        [0.1344, 0.0937, 0.1180, 0.2483, 0.1500, 0.1544, 0.1012, 0.0000],
        [0.1250, 0.0883, 0.0894, 0.1779, 0.1437, 0.1324, 0.1036, 0.1396]],
       grad_fn=<SelectBackward0>)

In [3]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)

mps
